<a href="https://colab.research.google.com/github/justverena/Circle_test_task/blob/main/notebooks/03_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate evaluate rouge_score

# Evaluation of Fine-Tuned Model

Now comparing the base model **Mistral-7B-Instruct** and the fine-tuned model with **LoRA adapter** on a set of unseen prompts.

I am generating responses from:
- Base model
- Fine-tuned model (QLoRA)

Then comparing outputs using:
- Qualitative analysis
- ROUGE evaluation

## Imports

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import pandas as pd
import evaluate

## Base model loading

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

## Finetuned model loading

In [ ]:
from google.colab import files
uploaded = files.upload()

In [7]:
ft_model = PeftModel.from_pretrained(base_model, "mistral-lora-adapter")

## Generation function

In [8]:
def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Test propmts

In [9]:
test_prompts = [
    "Explain what machine learning is in simple terms.",
    "What are the benefits of regular exercise?",
    "Describe the role of a database in software systems.",
    "What is climate change and why does it matter?",
    "How does the internet work?",
    "Explain object-oriented programming.",
    "What is overfitting in machine learning?",
    "Why is sleep important for health?",
    "What is a REST API?",
    "Explain how neural networks work."
]

## Generating responses

In [10]:
results = []

for prompt in test_prompts:
    base_answer = generate_response(base_model, prompt)
    ft_answer = generate_response(ft_model, prompt)

    results.append({
        "prompt": prompt,
        "base": base_answer,
        "fine_tuned": ft_answer
    })

for r in results:
  print("="*80)
  print("Prompt:", r["prompt"])
  print("\nBase Model:\n", r["base"])
  print("\nFine tuned model:\n", r["fine_tuned"])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o

Prompt: Explain what machine learning is in simple terms.

Base Model:
 Explain what machine learning is in simple terms.
Answer: Machine learning is a process that allows computers to learn from data, without being explicitly programmed by a human. Just as a human can learn from experience, a computer program can improve its performance by analyzing past results. Machine learning algorithms can be trained on historical data to discover patterns or relationships, and then use this information to make predictions or take actions in the future. For example, a machine learning algorithm could analyze website traffic to predict which products a customer is likely to buy, or it could learn to recognize a picture of a cat.

Fine tuned model:
 Explain what machine learning is in simple terms.
Answer: Machine learning is a type of artificial intelligence where a computer teaches and learns to do something on its own, with no human intervention. A computer could learn to recognize a picture of 

## Results table

In [11]:
df = pd.DataFrame(results)
df

,prompt,base,fine_tuned
0,Explain what machine learning is in simple terms.,Explain what machine learning is in simple ter...,Explain what machine learning is in simple ter...
1,What are the benefits of regular exercise?,What are the benefits of regular exercise?\nRe...,What are the benefits of regular exercise?\nAn...
2,Describe the role of a database in software sy...,Describe the role of a database in software sy...,Describe the role of a database in software sy...
3,What is climate change and why does it matter?,What is climate change and why does it matter?...,What is climate change and why does it matter?...
4,How does the internet work?,How does the internet work?\nAnswer: The inter...,How does the internet work?\nAnswer: The inter...
5,Explain object-oriented programming.,Explain object-oriented programming.\nAnswer: ...,Explain object-oriented programming.\nAnswer: ...
6,What is overfitting in machine learning?,What is overfitting in machine learning?\nAnsw...,What is overfitting in machine learning?\nAnsw...
7,Why is sleep important for health?,Why is sleep important for health?\nAnswer: Sl...,Why is sleep important for health?\nAnswer: Sl...
8,What is a REST API?,What is a REST API?\nAnswer: A REST (Represent...,What is a REST API?\nAnswer: A REST API is an ...
9,Explain how neural networks work.,Explain how neural networks work.\nAnswer: Neu...,Explain how neural networks work.\nAnswer: Neu...


## ROUGE Evaluation

In [16]:
rouge = evaluate.load("rouge")

base_texts = [r["base"] for r in results]
ft_texts = [r["fine_tuned"] for r in results]

rouge_scores = rouge.compute(predictions=ft_texts, references=base_texts)

print("ROUGE scores (Fine tuned vs Base Model):")
for k, v in rouge_scores.items():
    print(f"{k}: {v:.4f}")

ROUGE scores (Fine tuned vs Base Model):
rouge1: 0.4987
rouge2: 0.2435
rougeL: 0.3442
rougeLsum: 0.3839


## Conclusion
Based on the qualitative analysis of responses and ROUGE evaluation I can make the conclusion:

1. **Clarity and conciseness:** The fine-tuned model generally writes shorter and more concise answers compared to the base model.
2. **Details** The base model sometimes gives longer, more detailed explanations unlike the fine-tuned model, which can be good for instruction-following tasks but can miss some details.

3. **Consistency** The fine-tuned model uses more consistent phrasing because I prompted during fine-tuning: "Answer clearly and concisely".

4. **ROUGE evaluation:** The ROUGE scores between the fine-tuned and base model are:
   - ROUGE-1: 0.4987
   - ROUGE-2: 0.2435
   - ROUGE-L: 0.3442
   - ROUGE-Lsum: 0.3839  

   These numbers shows moderate lexical overlap, which is expected because the fine-tuned model is rewriting explanations with slightly different words and structure, but keeping the meaning.

Overall, the fine-tuning with LoRA made the model more suited for generating clear and structured answers and preserving the base model's knowledge.